### CONTENT-BASED MOVIE RECOMMENDATION SYSTEM
### Dataset: IMDB Genres (jquigl/imdb-genres)
### Method: TF-IDF + Cosine Similarity (No pre-trained embeddings)

SECTION 1: LOAD DATASET

In [1]:
from datasets import load_dataset

dataset = load_dataset("jquigl/imdb-genres")

train_data = dataset["train"]
val_data = dataset["validation"]

train_data[0]

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'movie title - year': 'Flaming Ears - 1992',
 'genre': 'Fantasy',
 'expanded-genres': 'Fantasy, Sci-Fi',
 'rating': 6.0,
 'description': 'Flaming Ears is a pop sci-fi lesbian fantasy feature set in the year 2700 in the fictive burned-out city of Asche. It follows the tangled lives of three women - Volley, Nun and Spy.'}

SECTION 2: CONVERT TO DATAFRAME

In [2]:
import pandas as pd

df = train_data.to_pandas()
val_df = val_data.to_pandas()

print("Train data shape: ", df.shape)
df.head(3)

Train data shape:  (238256, 5)


,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."


SECTION 3: EXPLORATORY DATA ANALYSIS (TRAIN SET)

In [6]:
# Check for null values, duplicates and NaN values in rating
print("Null counts:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("String NaN in rating:", (df['rating'] == 'NaN').sum())

Null counts:
 movie title - year        0
genre                     0
expanded-genres           0
rating                69721
description               0
dtype: int64

Duplicates: 18
String NaN in rating: 0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238256 entries, 0 to 238255
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movie title - year  238256 non-null  object 
 1   genre               238256 non-null  object 
 2   expanded-genres     238256 non-null  object 
 3   rating              168535 non-null  float64
 4   description         238256 non-null  object 
dtypes: float64(1), object(4)
memory usage: 9.1+ MB


SECTION 4: CLEAN TRAIN DATA

In [9]:
# Remove duplicate rows
df = df.drop_duplicates()
print("After dedup shape:", df.shape)

After dedup shape: (238238, 5)


In [ ]:
# Split 'movie title - year' column into separate title and year columns
df['title'] = df['movie title - year'].str.split(' -').str[0]
df['year']  = df['movie title - year'].str.split(' -').str[1]

In [11]:
df = df.drop('rating', axis=1)
df.head(0)

,movie title - year,genre,expanded-genres,description,title,year


In [16]:
# Combine description, genre, expanded-genres into a single 'tags' column
df['tags'] = df['description'] + df['genre'] + df['expanded-genres']

# Keep only the required columns for the recommender
movies = df[['title', 'tags']].copy()

In [18]:
movies.head()

,title,tags
0,Flaming Ears,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig,Six people - three couples - meet at random at...
2,Povjerenje,"In a small unnamed town, in year 2025, Krsto w..."
3,Gulliver Returns,The legendary Gulliver returns to the Kingdom ...
4,Prithvi Vallabh,"Seminal silent historical film, the story feat..."


In [17]:
print("Sample tags before cleaning:")
print(movies['tags'][1])

Sample tags before cleaning:
Six people - three couples - meet at random at a dance restaurant in the Copenhagen nightlife. A marriage swindler and former actress, an elderly Supreme Court attorney with wife as well as...                See full summary »RomanceComedy, Drama, Romance


SECTION 5: TEXT PREPROCESSING (TRAIN SET)

In [19]:
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [20]:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Function for Stemmer
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [32]:
# Full preprocessing pipeline
movies['tags'] = (
    movies['tags']
    .fillna("")
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .apply(lambda x: " ".join(
        [w for w in x.split() if w not in stop_words]
    ))
    .apply(stem)
)

movies['title'].str.lower()

0                  flaming ears
1                jeg elsker dig
2                    povjerenje
3              gulliver returns
4               prithvi vallabh
                  ...          
238251                kaalapani
238252               the set-up
238253    der bucklige von soho
238254         the shadow thing
238255        gatwick gangsters
Name: title, Length: 238238, dtype: object

In [33]:
print("Sample tags after cleaning:")
print(movies['tags'].iloc[1])

Sample tags after cleaning:
six peopl three coupl meet random danc restaur copenhagen nightlif marriag swindler former actress elderli suprem court attorney wife well see full summari romancecomedi drama romanc


SECTION 6: TF-IDF VECTORIZATION

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(
    max_features=15000,
    min_df=5,
    stop_words='english'
)

In [35]:
vectors = tf.fit_transform(movies['tags'])

In [36]:
print("TF-IDF matrix shape:", vectors.shape)
print("Sample features:", tf.get_feature_names_out()[1000:1010])

TF-IDF matrix shape: (238238, 15000)
Sample features: ['archivist' 'archnemesi' 'arctic' 'arden' 'ardenn' 'ardent' 'arduou'
 'area' 'arena' 'arent']


In [37]:
movies.head(3)

,title,tags,movie_id
0,Flaming Ears,flame ear pop scifi lesbian fantasi featur set...,0
1,Jeg elsker dig,six peopl three coupl meet random danc restaur...,1
2,Povjerenje,small unnam town year 2025 krsto work agenc of...,2


SECTION 7: ASSIGN MOVIE IDs ON TRAIN SET

In [38]:
movies['movie_id'] = range(len(movies))

movies[['movie_id', 'title', 'tags']].head(3)

,movie_id,title,tags
0,0,Flaming Ears,flame ear pop scifi lesbian fantasi featur set...
1,1,Jeg elsker dig,six peopl three coupl meet random danc restaur...
2,2,Povjerenje,small unnam town year 2025 krsto work agenc of...


SECTION 8: RECOMMENDATION FUNCTION

In [51]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_movies(input_text, top_k=5):
    """
    Accepts:
    - Movie Titles by matching with movie's TF-IDF vector
    - Any free text by converting to TF-IDF vector
    """
    input_text_lower = input_text.lower()

    # Step 1: Tries to locate exact title match in the train set
    matched = movies[movies['title'] == input_text_lower]

    if not matched.empty:
        # Found exact title. Matches with pre-computed vector
        movie_index = matched.index[0]
        input_vector = vectors[movie_index]
        ignore_index = {movie_index}
    else:
        # Not a title. A free text. Computes vector
        input_vector = tf.transform([input_text])
        ignore_index = set()

    # Step 2: Compute cosine similarity against all train movies
    sim_scores = cosine_similarity(input_vector, vectors).flatten()

    # Step 3: Sort indices from highest to lowest similarity
    sorted_indices = np.argsort(-sim_scores)

    # Step 4: Collect top 5 unique movies, skipping ignored index
    recommended = []
    seen_titles = set()
    for i in sorted_indices:
        title = movies.iloc[i]['title']
        if i not in ignore_index and title not in seen_titles:
            recommended.append(title)
            seen_titles.add(title)
        if len(recommended) >= topk:
            break

    return recommended

SECTION 9: PREPROCESS VALIDATION SET

In [52]:
val_df = val_data.to_pandas()
val_df = val_df.drop_duplicates()

In [53]:
# Get title and year
val_df['title'] = val_df['movie title - year'].str.split(' -').str[0]
val_df['year']  = val_df['movie title - year'].str.split(' -').str[1]

In [55]:
val_df['tags'] = val_df['description'] + val_df['genre'] + val_df['expanded-genres']

In [58]:
# Apply the data cleaning and stemming pipeline same as train data
val_df['tags'] = (
    val_df['tags']
    .fillna("")
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .apply(lambda x: " ".join(
        [w for w in x.split() if w not in stop_words]
    ))
    .apply(stem)
)
val_df['title'] = val_df['title'].str.lower()

In [61]:
val_df['movie_id'] = range(len(val_df))

val_df[['movie_id', 'title', 'tags']].head(3)

,movie_id,title,tags
0,0,the dictator,pursu cab driver unpaid fare millionair son br...
1,1,edsa xxx: nothing ever changes in the ever-cha...,polit realism absurdist music everyth film bas...
2,2,flaming waters,sever year absenc young sailor dan oneil retur...


In [ ]:
val_movies = val_df[['title', 'tags', 'rating', 'year', 'genre']].copy()
val_movies['movie_id'] = range(len(val_movies))

print("Validation set shape:", val_movies.shape)
val_movies.head(3)

Validation set shape: (29808, 6)


,title,tags,rating,year,genre,movie_id
0,the dictator,pursu cab driver unpaid fare millionair son br...,2.6,1922,Adventure,0
1,edsa xxx: nothing ever changes in the ever-cha...,polit realism absurdist music everyth film bas...,4.6,2014,Fantasy,1
2,flaming waters,sever year absenc young sailor dan oneil retur...,7.0,1925,Action,2


SECTION 10: EVALUATION | MANUAL SPOT CHECK ON VALIDATION SET

In [ ]:
print("EVALUATION | Manual Spot Check on Validation Set")

# Pick 5 random movies from the validation set to test
test_movies = val_movies.sample(n=5, random_state=42)

for _, row in test_movies.iterrows():
    print(f"\nQuery movie : {row['title']}")
    print(f"Genre       : {row['genre']}")
    print(f"Recommended :")
    results = recommend_movies(row['title'], top_k=5)
    for i, title in enumerate(results, 1):
        print(f"  {i}. {title}")

SECTION 11: TESTING | MANUAL RECOMMENDATION QUERIES

In [ ]:
print("\n" + "="*50)
print("RECOMMENDATION TESTS")
print("="*50)

# Test 1: Free-text genre keyword
print("\n[Test 1] Query: 'romance'")
print(recommend_movies('romance'))

# Test 2: Descriptive keywords
print("\n[Test 2] Query: 'action thriller spy'")
print(recommend_movies('action thriller spy'))

# Test 3: Exact movie title (will use its own vector)
print("\n[Test 3] Query: exact movie title from train set")
sample_title = movies['title'].iloc[0]
print(f"  Title: '{sample_title}'")
print(recommend_movies(sample_title))

# Test 4: A movie from the validation set (unseen during training)
print("\n[Test 4] Query: movie title from validation set")
val_sample_title = val_movies['title'].iloc[0]
print(f"  Title: '{val_sample_title}'")
print(recommend_movies(val_sample_title))

# Test 5: Description-style query
print("\n[Test 5] Query: 'a young wizard discovers magical powers'")
print(recommend_movies('a young wizard discovers magical powers'))



RECOMMENDATION TESTS

[Test 1] Query: 'romance'
['Once Upon a Time in the Midwest', 'A Man Called Blade', 'En flicka för mej', 'Okey, Mister Pancho', 'Love and Other Disasters']

[Test 2] Query: 'action thriller spy'
['Sanctuary', 'JacobMarc26: The Movie', 'Kurukshetramu', 'Underverden II', 'Fighting Warays sa Ilokos']

[Test 3] Query: exact movie title from train set
  Title: 'Flaming Ears'
['Flaming Ears', 'Love(less)', 'Elliptica', 'Contessa', 'Zombified']

[Test 4] Query: movie title from validation set
  Title: 'The Dictator'
['Octonauts & the Great Barrier Reef', 'Lords of the Deep', 'Americano', 'The Navigator', 'To fluer i ett smekk']

[Test 5] Query: 'a young wizard discovers magical powers'
['Aggi Meeda Guggilam', 'Eko Eko Azarak II: Birth of the Wizard', 'Where Do We Go from Here?', 'Wizards', 'Gekijouban Mahou sensei Negima! Anime Final']
